# IGCRN: reproducible public-data preparation on Colab T4
Select Runtime → Change runtime type → T4 GPU. Default: prepare a small public-data smoke set, simulate mixtures, save manifests. **Full training is disabled.** Mini LibriSpeech requires one 332 MB archive; only selected ESC-10 noise WAVs are fetched. No train-clean-100 or full ESC-50 download.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO = Path('/content/IGCRN-DualMic-Speech-Enhancement')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/ZhengchenYuan/IGCRN-DualMic-Speech-Enhancement.git', str(REPO)], check=True, cwd='/content')
os.chdir(REPO)
assert (REPO / 'scripts' / '__init__.py').is_file()

def run(*args):
    # Pin the working directory for every Python command, even after another cell changes it.
    subprocess.run([sys.executable, *map(str, args)], cwd=REPO, check=True)

run('-m', 'pip', 'install', '-r', 'requirements.txt')
run('-m', 'pip', 'install', '-e', '.')
print('Repository root:', REPO)


In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable T4 GPU first'
print(torch.cuda.get_device_name(0))
subprocess.run(['nvidia-smi'], cwd=REPO, check=True)
# Invoke pytest through the same interpreter from the repository root.
subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=REPO, check=True)


## Download and prepare public recordings
Licenses and upstream per-clip attribution are checked before publishing the catalog. A missing/unsupported license stops preparation; do not fill in assumed metadata. `smoke` selects 1/1/1 disjoint speakers, up to 2 utterances each, and 2/1/1 noise recordings. `experiment` selects 12/3/3 speakers, up to 20 utterances each, and 40/10/10 noise recordings. Counts can be overridden with the CLI. Use a new output directory when changing mode, counts or seed.

In [ ]:
MODE = 'smoke'  # Change to 'experiment' only when ready for a larger preparation.
SEED = 2026
DATA = Path('/content') / f'igcrn_data_{MODE}'
SAVE = Path('/content/drive/MyDrive/IGCRN_NIO')
run('-m', 'scripts.prepare_data', '--mode', MODE, '--seed', SEED, '--output', DATA)
run('-m', 'scripts.simulate', '--catalog', DATA / 'catalog.jsonl', '--output', DATA / 'mixtures',
    '--count', 9 if MODE == 'smoke' else 300, '--seconds', 2, '--seed', SEED)


## Preserve important artifacts only
Raw archives, selected recordings, RIR arrays and generated mixtures remain under `/content/`. Drive stores the catalog, source/license evidence, preparation record, frozen mixture manifests, software environment and commit. Reruns refuse to overwrite different saved artifacts. After a Colab reset, reproduce sources and mixtures and compare their hashes before resuming checkpoints.

In [ ]:
import shutil
ARTIFACTS = SAVE / 'manifests' / f'{MODE}-seed{SEED}'
ARTIFACTS.mkdir(parents=True, exist_ok=True)

def preserve(source, destination):
    if destination.exists() and destination.read_bytes() != source.read_bytes():
        raise RuntimeError(f'Saved artifact differs: {destination}; use a new run directory and investigate.')
    shutil.copy2(source, destination)

for name in ('catalog.jsonl', 'preparation.json'):
    preserve(DATA / name, ARTIFACTS / name)
for split in ('train', 'valid', 'test'):
    preserve(DATA / 'mixtures' / f'{split}.jsonl', ARTIFACTS / f'{split}.jsonl')
shutil.copytree(DATA / 'evidence', ARTIFACTS / 'evidence', dirs_exist_ok=True)
for filename, command in [('environment.txt', [sys.executable, '-m', 'pip', 'freeze']),
                          ('commit.txt', ['git', 'rev-parse', 'HEAD'])]:
    with (ARTIFACTS / filename).open('w') as stream:
        subprocess.run(command, cwd=REPO, stdout=stream, check=True)
print('Saved metadata only:', ARTIFACTS)


## Optional later training and evaluation (disabled)
Leave the switch below false for data preparation. Training requires `MODE = "experiment"`; both arms must use the same frozen data and budget. Checkpoints go to `/content/drive/MyDrive/IGCRN_NIO/dual` and `duplicate`. For interrupted runs add `--resume <arm>/last.pt` to that arm’s command. Review memory needs and the README before enabling.

In [ ]:
RUN_FULL_TRAINING = False
if RUN_FULL_TRAINING:
    assert MODE == 'experiment', 'Use the experiment dataset for research training'
    for arm in ('dual', 'duplicate'):
        run('-m', 'scripts.train', '--config', 'configs/small.yaml', '--data', DATA / 'mixtures',
            '--mode', arm, '--output', SAVE / arm, '--batch-size', 1)
    run('-m', 'scripts.evaluate', '--manifest', DATA / 'mixtures' / 'test.jsonl',
        '--dual', SAVE / 'dual' / 'last.pt', '--duplicate', SAVE / 'duplicate' / 'last.pt',
        '--output', SAVE / 'evaluation')
else:
    print('Preparation complete. Full training was not launched.')
